In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/snm3C/higashi'

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/embedding/higashi_filtered.h5ad')
select = ['line', 'time']
adata.obs = adata.obs[select]
adata.var.index = [f'feature{x}' for x in adata.var.index]

In [ ]:
adata

In [ ]:
higashi = pd.read_csv(f'{projdir}/csv/A01b_higashi_embeds_100kb.csv', index_col=0)
n_pcs = 50
select = [f'embed{x}' for x in range(1, n_pcs+1)]
higashi_pcs = higashi[select]
adata.obsm['X_pca'] = higashi_pcs.reindex(adata.obs.index).values

In [ ]:
sc.pl.umap(adata, color=['line', 'time'])

In [ ]:
n_pcs = 20
k = 15
sc.pp.neighbors(adata, n_neighbors=k, n_pcs=n_pcs, use_rep='X_pca', random_state=0)

In [ ]:
resolution = 1.0
sc.tl.leiden(adata, resolution=resolution, random_state=0)

In [ ]:
sc.pl.umap(adata, color=['time', 'line', 'leiden'], ncols=2)

In [ ]:
sc.pl.umap(adata, color=['time', 'leiden'], ncols=2)

In [ ]:
sc.pl.umap(adata, color=['leiden'], groups = '2')

In [ ]:
sc.pl.umap(adata, color=['leiden'], groups = '8')

In [ ]:
sc.pl.umap(adata, color=['leiden'], groups = '13')

In [ ]:
sc.pl.umap(adata, color=['leiden'], groups = '14')

In [ ]:
adata.obs['leiden'].value_counts()

### check which clusters have more IPS

In [ ]:
indir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/snmCT/mc'
clusters = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['cluster']

In [ ]:
adata

In [ ]:
s = '2'
tmp_meta = adata.obs[adata.obs['leiden'] == s]
tmp_meta['cluster'].value_counts() / tmp_meta['cluster'].value_counts().sum()

In [ ]:
s = '4'
tmp_meta = adata.obs[adata.obs['leiden'] == s]
tmp_meta['cluster'].value_counts() / tmp_meta['cluster'].value_counts().sum()

In [ ]:
s = '8'
tmp_meta = adata.obs[adata.obs['leiden'] == s]
tmp_meta['cluster'].value_counts() / tmp_meta['cluster'].value_counts().sum()

In [ ]:
s = '13'
tmp_meta = adata.obs[adata.obs['leiden'] == s]
tmp_meta['cluster'].value_counts() / tmp_meta['cluster'].value_counts().sum()

In [ ]:
s = '14'
tmp_meta = adata.obs[adata.obs['leiden'] == s]
tmp_meta['cluster'].value_counts() / tmp_meta['cluster'].value_counts().sum()

### rename clusters

In [ ]:
adata.obs.loc[mask,'leiden']

In [ ]:
whitelist = ['2', '8', '13', '14']
rename_dict = {'2' : 'IPS_main', '8' : 'IPS_alt',
               '13' : 'IPS_main', '14' : 'IPS_alt'}
adata.obs['renamed_leiden'] = adata.obs['leiden'].astype(str)

mask = [x in whitelist for x in adata.obs['leiden']]
adata.obs.loc[mask,'renamed_leiden'] = [rename_dict.get(x) for x in adata.obs.loc[mask,'leiden']]
adata.obs['renamed_leiden'].value_counts()

In [ ]:
sc.pl.umap(adata, color=['time', 'line', 'renamed_leiden'], ncols=2)

### how about we make a new annotation manually removing pre-D9 IPS's

In [ ]:
time = 12
adata.obs['ips_cluster_time'] = 'non-IPS'

for s in ['IPS_main', 'IPS_alt']:
    mask1 = adata.obs['renamed_leiden'] == s
    mask2 = adata.obs['time'] >= time
    mask = np.logical_and(mask1, mask2)
    adata.obs.loc[mask,'ips_cluster_time'] = s

In [ ]:
ips_main_df = pd.DataFrame(adata.obs[adata.obs['renamed_leiden'] == 'IPS_main']['time'].value_counts().sort_index())
ips_main_df.index.name = 'time'
ips_main_df.columns = ['IPS_main']
ips_main_df['IPS_alt'] = adata.obs[adata.obs['renamed_leiden'] == 'IPS_alt']['time'].value_counts().sort_index()
ips_main_df.fillna(0).astype(int)

In [ ]:
adata.obs['ips_cluster_time'].value_counts(),  \
adata.obs['renamed_leiden'].value_counts()

In [ ]:
# mask = [x.split('_')[0] == 'IPS' for x in adata.obs['renamed_leiden']]
# final_meta = adata.obs[mask]
# final_meta.shape

In [ ]:
# final_meta.to_csv(f'{projdir}/csv/clusters/ips_main_alt_leiden.csv', sep='\t')

In [ ]:
adata.obs.to_csv(f'{projdir}/csv/clusters/leiden_with_ips_main_alt.csv', sep='\t')

### the main cluster is bothering me a little bit

In [ ]:
!date